# ETL de la PUI: autónomo y explicado paso a paso

Este cuaderno reúne aquí mismo las funciones de transformación: no importa `config.py` ni módulos `src/*.py` y no descarga desde Drive. Lee archivos locales, muestra un snapshot tras cada operación importante y mantiene una auditoría de tamaños y nulos.

El flujo integra coincidencias administrativas de la PUI, confrontas manuales, el maestro RNPDNO y defunciones por institución. Las muestras ocultan FUB, CURP y otros datos identificadores. La salida solo se exporta si activas `EXPORTAR_RESULTADO` en la configuración.

Ejecuta las celdas en orden. Verifica las rutas locales en la primera sección antes de iniciar.

## 1. Preparación y rutas locales

Aquí se fijan los cuatro insumos, la fecha del corte (se deriva del nombre del CSV), la salida y las opciones de inspección. El CSV se toma explícitamente del corte más reciente disponible localmente, pero la ruta concreta queda visible y editable.

In [ ]:
# CONFIGURACIÓN: rutas locales y funciones del proceso definidas en este notebook.
from pathlib import Path
from datetime import date, datetime
from collections import Counter
import re
import unicodedata
import warnings
import numpy as np
import pandas as pd
from IPython.display import display

# Descubre la raíz a partir de marcadores del proyecto; funciona desde cualquier
# subcarpeta dentro del repositorio, incluida `notebooks luis/`.
_DIRECTORIOS_BUSQUEDA = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PROJECT_ROOT = next(
    (p for p in _DIRECTORIOS_BUSQUEDA
     if (p / "input" / "pui").is_dir()
     and (p / "output" / "Resultado_1_Procesamiento_ETL_RNPDNO.parquet").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto: se esperaban input/pui/ y "
        "output/Resultado_1_Procesamiento_ETL_RNPDNO.parquet."
    )

PUI_DIR = PROJECT_ROOT / "input" / "pui"
# Actualiza esta constante al corte diario más reciente disponible.
RUTA_COINCIDENCIAS = PUI_DIR / "reporte_coincidencias_validas_2026-10-19.csv"
RUTA_REGISTROS = PUI_DIR / "registros_con_pui_13032026.xlsx"
RUTA_DEFUNCIONES = PUI_DIR / "tabla_defunciones_por_institucion_05052026.xlsx"
RUTA_CURPS_EXCLUIDAS = PUI_DIR / "curps_excluidas.txt"
RUTA_COMPRIMIDO = PROJECT_ROOT / "output" / "Resultado_1_Procesamiento_ETL_RNPDNO.parquet"
RUTA_SALIDA = PROJECT_ROOT / "output" / "Resultado_3_Procesamiento_ETL_PUI_autonomo.parquet"
EXPORTAR_RESULTADO = True
MOSTRAR_DATOS_SENSIBLES = False
RUTA_REFERENCIA_R = None
AUDITORIA = []

COLUMNAS_COINCIDENCIAS = [
    "institucion_id", "institucion", "fase_busqueda", "fub", "curp", "tipo_evento",
    "fecha_evento", "fecha_desaparicion", "entidad_federativa", "reporte_created_at",
    "coincidencia_id", "calle", "codigo_postal", "colonia", "direccion", "is_valid",
]
MAP_COINCIDENCIAS = {c: [c] for c in COLUMNAS_COINCIDENCIAS}
MAP_REGISTROS = {
    "fub": ["fub"], "curp_victima": ["curp_victima", "curp"],
    "indicadora_indicio_de_vida": ["indicadora_indicio_de_vida"],
    "institucion_indicadora_de_vida": ["institucion_indicadora_de_vida"],
    "numero_instituciones": ["numero_instituciones"],
}
MAP_DEFUNCIONES = {"fub": ["fub"], "tipo_defuncion_institucion": ["tipo_defuncion_institucion"]}
MAP_COMPRIMIDO = {
    "fub": ["fub_rnpdno"], "curp_victima": ["curp_vic_derivada_rnpdno"],
    "clasificacion_des": ["clasificacion_des_derivada_rnpdno"],
    "fecha_hechos_modificada": ["fecha_hechos_modificada_derivada_rnpdno"],
}
COLUMNAS_SALIDA = [
    "fub", "curp_victima", "indicadora_indicio_de_vida", "institucion_indicadora_de_vida",
    "numero_instituciones", "indicadora_coincidencia_manual", "tipo_tramites",
    "tramites_repetidos", "fecha_evento", "clasificacion_des",
]
NOMBRES_EXPORTADOS = {
    "fub": "fub_derivada_pui",
    "curp_victima": "curp_victima_derivada_rnpdno_pui",
    "indicadora_indicio_de_vida": "indicadora_indicio_de_vida_derivada_pui",
    "institucion_indicadora_de_vida": "institucion_indicadora_de_vida_derivada_pui",
    "numero_instituciones": "numero_instituciones_derivada_pui",
    "indicadora_coincidencia_manual": "indicadora_coincidencia_manual_derivada_pui",
    "tipo_tramites": "tipo_tramites_derivada_pui",
    "tramites_repetidos": "tramites_repetidos_derivada_pui",
    "fecha_evento": "fecha_evento_derivada_pui",
    "clasificacion_des": "clasificacion_des_derivada_rnpdno",
}
ABREVIATURAS_INSTITUCIONES = {
    r"Comisión Nacional del Sistema de Ahorro para el Retiro \(CONSAR\)": "CONSAR",
    r"Comunicaciones y Transportes \(SICT\)": "SICT",
    r"INSTITUTO MEXICANO DEL SEGURO SOCIAL": "IMSS-PUI",
    r"Instituto Nacional de Migración \(INM\)": "INAMI",
    r"Petróleos Mexicanos \(PEMEX\)": "PEMEX",
    r"Secretaría de Bienestar \(SEBIEN\)": "SEBIEN",
    r"Secretaría de Infraestructura": "SICT",
    r"Secretaría de Relaciones Exteriores \(SRE\)": "SRE",
    r"Servicio de Administración Tributaria \(SAT\)": "SAT",
    r"Servicios de Salud del Instituto Mexicano del Seguro Social para el Bienestar \(IMSS-Bienestar\)": "IMSS-Bienestar",
}

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"CSV PUI: {RUTA_COINCIDENCIAS.name}")
print("Este cuaderno es local/autocontenido; no importa código del proyecto.")

In [ ]:
# Verifica todas las rutas de entrada sin cambiar la configuración de la primera celda.
INSUMOS_LOCALES = {
    "Coincidencias PUI": RUTA_COINCIDENCIAS,
    "Registros PUI": RUTA_REGISTROS,
    "Defunciones por institución": RUTA_DEFUNCIONES,
    "CURP excluidas": RUTA_CURPS_EXCLUIDAS,
    "RNPDNO procesado": RUTA_COMPRIMIDO,
}
INSUMOS_FALTANTES = [nombre for nombre, ruta in INSUMOS_LOCALES.items() if not ruta.is_file()]
for nombre, ruta in INSUMOS_LOCALES.items():
    print(f"{'OK' if ruta.is_file() else 'FALTA'} | {nombre}: {ruta}")
if INSUMOS_FALTANTES:
    raise FileNotFoundError("Faltan insumos locales: " + ", ".join(INSUMOS_FALTANTES))

## 2. Funciones auxiliares: semántica de R, carga y auditoría

Algunas agregaciones del original de R dependen del orden y del tratamiento de nulos. Estas funciones conservan ese comportamiento en pandas (`first`, únicos en orden, conteos, reemplazos secuenciales). Los snapshots enmascaran campos personales y guardan conteos sin almacenar copias completas.

In [ ]:
# 2. FUNCIONES REUTILIZABLES DEFINIDAS DENTRO DEL NOTEBOOK.
def clean_names(df):
    """Equivalente mínimo a janitor::clean_names para las columnas de entrada."""
    out = df.copy()
    out.columns = [re.sub(r"_+", "_", re.sub(r"[^0-9a-zA-Z]+", "_", str(c).strip())).strip("_").lower() for c in out.columns]
    return out


def primero(serie):
    """Primer elemento del grupo incluso cuando es nulo (dplyr::first)."""
    return serie.iloc[0] if len(serie) else np.nan


def paste_unique(serie, sep=", "):
    """Únicos en orden de aparición, como paste(unique(x), collapse=sep)."""
    return sep.join(dict.fromkeys(serie.astype(str)))


def paste_sorted_unique(serie, sep=", "):
    return sep.join(sorted(dict.fromkeys(serie.astype(str))))


def r_str_replace_all(serie, reemplazos):
    """Aplica sustituciones secuencialmente (la salida de una entra en la siguiente)."""
    salida = serie.copy()
    for patron, valor in reemplazos.items():
        salida = salida.str.replace(patron, valor, regex=True)
    return salida


def conteos_en_orden_de_aparicion(texto):
    if not isinstance(texto, str):
        return np.nan
    partes = texto.split(", ")
    conteos = Counter(partes)
    return ", ".join(str(conteos[p]) for p in dict.fromkeys(partes))


def unicos_en_orden_de_aparicion(texto):
    if not isinstance(texto, str):
        return np.nan
    return ", ".join(dict.fromkeys(texto.split(", ")))


def comma(valor):
    return f"{valor:,}"


def enmascarar(df, n=5):
    muestra = df.head(n).copy()
    if not MOSTRAR_DATOS_SENSIBLES:
        sensibles = re.compile(r"fub|curp|nombre|apellido|folio|direccion|calle|colonia|codigo_postal", re.I)
        for columna in muestra.columns:
            if sensibles.search(str(columna)):
                muestra[columna] = muestra[columna].map(lambda x: "[OCULTO]" if pd.notna(x) else x)
    return muestra


def snapshot(etapa, df, mostrar=True):
    fila = {
        "etapa": etapa, "filas": len(df), "columnas": df.shape[1],
        "nulos": int(df.isna().sum().sum()),
        "memoria_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2),
    }
    AUDITORIA.append(fila)
    print(f"\n[{etapa}] {len(df):,} filas × {df.shape[1]} columnas | {fila['memoria_mb']} MB")
    print("Tipos (primeras 12):", df.dtypes.astype(str).head(12).to_dict())
    nulos = df.isna().sum().sort_values(ascending=False)
    if (nulos > 0).any():
        print("Más nulos:", nulos[nulos > 0].head(6).to_dict())
    if mostrar:
        display(enmascarar(df))
    return fila


def leer_csv_pui(ruta):
    """Intenta UTF-8 y luego Latin-1, todo como texto para preservar ceros iniciales."""
    try:
        return pd.read_csv(ruta, dtype=str, low_memory=False, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(ruta, dtype=str, low_memory=False, encoding="latin-1")


def leer_excel_mapeado(ruta, mapping, hoja=0):
    df = pd.read_excel(ruta, sheet_name=hoja)
    cols_norm = {clean_names(pd.DataFrame(columns=[c])).columns[0]: c for c in df.columns}
    renombrar = {}
    faltan = []
    for destino, aliases in mapping.items():
        encontrado = next((cols_norm.get(clean_names(pd.DataFrame(columns=[a])).columns[0]) for a in aliases
                           if clean_names(pd.DataFrame(columns=[a])).columns[0] in cols_norm), None)
        if encontrado:
            renombrar[encontrado] = destino
        else:
            faltan.append(destino)
    if faltan:
        raise ValueError(f"En {Path(ruta).name} faltan columnas {faltan}; presentes: {list(df.columns)}")
    return df.rename(columns=renombrar)[list(mapping)].copy()


def leer_parquet_mapeado(ruta, mapping):
    df = pd.read_parquet(ruta)
    faltan = [alias[0] for destino, alias in mapping.items() if alias[0] not in df.columns]
    if faltan:
        raise ValueError(f"En {Path(ruta).name} faltan columnas {faltan}")
    return df[[alias[0] for alias in mapping.values()]].rename(
        columns={aliases[0]: destino for destino, aliases in mapping.items()}
    ).copy()


def verificar_archivos():
    archivos = [RUTA_COINCIDENCIAS, RUTA_REGISTROS, RUTA_DEFUNCIONES, RUTA_COMPRIMIDO]
    faltan = [str(x) for x in archivos if not x.exists()]
    if faltan:
        raise FileNotFoundError("Faltan insumos locales requeridos:\n- " + "\n- ".join(faltan))
    m = re.search(r"(\d{4})-(\d{2})-(\d{2})", RUTA_COINCIDENCIAS.name)
    if not m:
        raise ValueError(f"El nombre del CSV debe incluir fecha YYYY-MM-DD: {RUTA_COINCIDENCIAS.name}")
    anio, mes, dia = map(int, m.groups())
    return date(anio, mes, dia).strftime("%d%m%Y")


FECHA_CORTE = verificar_archivos()
print(f"Funciones listas. Corte de esta corrida: {FECHA_CORTE}")

## 3. Lectura de coincidencias administrativas

Se lee el CSV como texto (para no perder ceros a la izquierda de `institucion_id`), se limpian encabezados y se comprueba que estén presentes las columnas requeridas. El primer snapshot permite ver el volumen y la estructura antes de aplicar reglas.

In [ ]:
# 3. LEER Y VALIDAR EL CSV LOCAL DE COINCIDENCIAS.
coincidencias = leer_csv_pui(RUTA_COINCIDENCIAS)
coincidencias = clean_names(coincidencias)
faltan_csv = [c for c in COLUMNAS_COINCIDENCIAS if c not in coincidencias.columns]
if faltan_csv:
    raise ValueError(f"Al CSV de coincidencias le faltan columnas: {faltan_csv}")
print(f"Archivo: {RUTA_COINCIDENCIAS.name} | Corte: {FECHA_CORTE}")
snapshot("01 - coincidencias leídas", coincidencias)

## 4. Validaciones de fase, CURP e indicador lógico

Solo se admiten fases de búsqueda 2 y 3; una fase distinta detiene el flujo. Las CURP vacías generan aviso, no detención. `is_valid` se convierte de texto a booleano para que los grupos activos/inactivos se calculen correctamente.

In [ ]:
# 4. VALIDAR FASES Y NORMALIZAR is_valid.
fase_num = pd.to_numeric(coincidencias["fase_busqueda"], errors="coerce")
if (~fase_num.isin([2, 3])).any():
    invalidas = coincidencias.loc[~fase_num.isin([2, 3]), "fase_busqueda"].value_counts(dropna=False)
    raise ValueError(f"Fases de búsqueda fuera de 2 y 3:\n{invalidas}")
print("Fases válidas:", fase_num.value_counts().sort_index().to_dict())
n_curp_vacias = int(coincidencias["curp"].isna().sum())
if n_curp_vacias:
    warnings.warn(f"Se encontraron {n_curp_vacias:,} CURP vacías.")

coincidencias["is_valid"] = coincidencias["is_valid"].map({
    "True": True, "true": True, "TRUE": True, "t": True,
    "False": False, "false": False, "FALSE": False, "f": False,
}).astype("boolean")
if coincidencias["is_valid"].isna().any():
    warnings.warn("Algunos valores de is_valid no se reconocieron como booleanos.")
print("is_valid:", coincidencias["is_valid"].value_counts(dropna=False).to_dict())
snapshot("02 - validaciones aplicadas", coincidencias, mostrar=False)

## 5. Limpiar FUB, excluir CURP señaladas y filtrar eventos RENAPO

El FUB de la coincidencia incluye al final el UUID de esa coincidencia; se recorta ese sufijo. Luego se eliminan las CURP contenidas en el archivo local de exclusiones. Para RENAPO (`institucion_id == '0044'`) se excluyen actas de nacimiento y eventos que contienen `|1`; las demás coincidencias se mantienen. Al final se añade el nombre de institución al tipo de evento.

In [ ]:
# 5. APLICAR LIMPIEZA DE COINCIDENCIAS. Cada contador permite seguir filas eliminadas.
n_antes_limpieza = len(coincidencias)
coincidencias = coincidencias.rename(columns={"fub": "fub_uuid"})
coincidencias["fub"] = (
    coincidencias["fub_uuid"].str.replace(r"([0-9A-F]-?){32}-?$", "", regex=True)
    .str.replace(r"-$", "", regex=True)
)

if RUTA_CURPS_EXCLUIDAS.exists():
    curps_excluidas = RUTA_CURPS_EXCLUIDAS.read_text(encoding="utf-8").split()
    for curp_excluida in curps_excluidas:
        coincidencias = coincidencias[coincidencias["curp"].ne(curp_excluida).fillna(False)]
    print(f"CURP excluidas por lista: {len(curps_excluidas)} valores; filas actuales: {len(coincidencias):,}")
else:
    warnings.warn(f"No se encontró la lista local de exclusiones: {RUTA_CURPS_EXCLUIDAS}")

renapo = coincidencias["institucion_id"].eq("0044").fillna(False)
es_acta = coincidencias["tipo_evento"].str.contains("Tipo de Acto: Nacimiento", regex=False, na=True)
antes_renapo = len(coincidencias)
coincidencias = coincidencias[~(renapo & es_acta)]
renapo = coincidencias["institucion_id"].eq("0044").fillna(False)
tiene_pipe_uno = coincidencias["tipo_evento"].str.contains(r"\|1", regex=True, na=True)
coincidencias = coincidencias[~(renapo & tiene_pipe_uno)]
# Replicar paste0: los nulos se vuelven el literal "NA".
coincidencias["tipo_evento"] = (
    coincidencias["tipo_evento"].fillna("NA") + " (" + coincidencias["institucion"].fillna("NA") + ")"
)
print(f"Coincidencias: {n_antes_limpieza:,} antes; {len(coincidencias):,} después; "
      f"{n_antes_limpieza-len(coincidencias):,} eliminadas en total.")
snapshot("03 - limpieza de coincidencias", coincidencias)

## 6. Agrupar coincidencias en una fila por FUB

La agrupación resume apariciones, instituciones, eventos, fases y direcciones. `primero()` conserva incluso un valor nulo inicial, como `dplyr::first()`. Se muestran las dimensiones antes/después para hacer visible cuánto reduce la granularidad.

In [ ]:
# 6. AGREGAR POR FUB, RESUMIR TRÁMITES Y CONTAR REPETICIONES.
coincidencias_agrupado = (
    coincidencias.groupby("fub", sort=True, dropna=False)
    .agg(
        no_apariciones=("fub", "size"),
        curp=("curp", primero),
        is_valid=("is_valid", primero),
        instituciones=("institucion", paste_unique),
        tipo_evento=("tipo_evento", lambda s: ", ".join(s.astype(str))),
        no_instituciones=("institucion_id", lambda s: s.nunique(dropna=False)),
        fases_busqueda=("fase_busqueda", paste_sorted_unique),
        instituciones_ids=("institucion_id", paste_unique),
        fecha_desaparicion=("fecha_desaparicion", primero),
        fecha_evento=("fecha_evento", primero),
        calle=("calle", primero), colonia=("colonia", primero),
        codigo_postal=("codigo_postal", primero), direccion=("direccion", primero),
    )
    .reset_index()
    .rename(columns={"is_valid": "desaparicion_activa"})
)
print(f"Coincidencias agrupadas: {len(coincidencias):,} filas fuente → {len(coincidencias_agrupado):,} FUB.")
snapshot("04 - coincidencias agrupadas", coincidencias_agrupado)

In [ ]:
# 7. SEPARAR EVENTOS ÚNICOS Y CALCULAR VECES REPETIDAS.
coincidencias_agrupado["tramites_repetidos"] = coincidencias_agrupado["tipo_evento"].map(conteos_en_orden_de_aparicion)
coincidencias_agrupado["tipo_evento"] = coincidencias_agrupado["tipo_evento"].map(unicos_en_orden_de_aparicion)
num_activas = int((coincidencias_agrupado["desaparicion_activa"] == True).sum())
num_inactivas = int((coincidencias_agrupado["desaparicion_activa"] == False).sum())
print(f"Desapariciones activas: {num_activas:,} | inactivas: {num_inactivas:,} | FUB agrupados: {len(coincidencias_agrupado):,}")
snapshot("05 - trámites resumidos", coincidencias_agrupado)

## 8. Cargar confrontas manuales, RNPDNO y defunciones

Las confrontas manuales y defunciones se leen de sus Excel locales. Del maestro RNPDNO se toman solo FUB, CURP derivada (imputada), clasificación y fecha de hechos modificada. El snapshot de cada entrada ayuda a comprobar que los cruces usan las llaves y tamaños esperados.

In [ ]:
# 8. LEER LOS TRES INSUMOS RESTANTES CON VALIDACIÓN DE COLUMNAS.
registros = leer_excel_mapeado(RUTA_REGISTROS, MAP_REGISTROS, hoja=0)
comprimido = leer_parquet_mapeado(RUTA_COMPRIMIDO, MAP_COMPRIMIDO)
comprimido["fecha_hechos_modificada"] = pd.to_datetime(comprimido["fecha_hechos_modificada"], errors="coerce")
defunciones = leer_excel_mapeado(RUTA_DEFUNCIONES, MAP_DEFUNCIONES, hoja=0)

print("Tamaños de insumos:")
for nombre_df, tabla in [("registros manuales", registros), ("maestro RNPDNO", comprimido), ("defunciones", defunciones)]:
    print(f"  {nombre_df}: {len(tabla):,} × {tabla.shape[1]}")
    snapshot(f"08 - lectura {nombre_df}", tabla, mostrar=False)

## 9. Cruce con confrontas manuales: primero FUB, luego CURP + fecha

Las coincidencias se intentan asociar a registros manuales primero por FUB. Las que no empatan se comparan por la combinación CURP y fecha de desaparición, lo que permite detectar a una misma persona aunque tenga distinto FUB. Se conservan las dos ramas y se registra cómo se obtuvo cada coincidencia.

In [ ]:
# 9. DOS ETAPAS DE MATCH: FUB Y, PARA LOS PENDIENTES, CURP+FECHA.
registros_manual = (
    registros[registros["indicadora_indicio_de_vida"] == "con registro administrativo"]
    .assign(indicadora_manual="con coincidencia manual")
    .rename(columns={
        "indicadora_indicio_de_vida": "indicadora_indicio_de_vida_manual",
        "institucion_indicadora_de_vida": "institucion_indicadora_de_vida_manual",
        "numero_instituciones": "numero_instituciones_rnpdno_manual",
    })[["fub", "indicadora_indicio_de_vida_manual", "institucion_indicadora_de_vida_manual",
       "numero_instituciones_rnpdno_manual", "indicadora_manual"]]
)
coincidencias_agrupado = coincidencias_agrupado.merge(registros_manual, on="fub", how="left")
coincidencias_agrupado["indicadora_manual"] = coincidencias_agrupado["indicadora_manual"].fillna("sin coincidencia manual")
sin_manual_fase1 = coincidencias_agrupado[
    coincidencias_agrupado["indicadora_manual"] == "sin coincidencia manual"
].copy()
print(f"Empates manuales por FUB: {(coincidencias_agrupado['indicadora_manual'] == 'con coincidencia manual').sum():,}")
print(f"Pendientes para cruce alternativo: {len(sin_manual_fase1):,}")
snapshot("09 - coincidencias después del cruce FUB", coincidencias_agrupado, mostrar=False)

# El cruce manual incorpora la fecha de hechos desde RNPDNO para armar llave CURP+fecha.
registros = registros[registros["indicadora_indicio_de_vida"] == "con registro administrativo"].merge(
    comprimido[["fub", "fecha_hechos_modificada"]], on="fub", how="left"
)
registros["curp_fecha_desaparicion"] = np.where(
    registros["curp_victima"].isna(), np.nan,
    registros["curp_victima"].astype(str) + "_" + registros["fecha_hechos_modificada"].dt.strftime("%Y-%m-%d"),
)
sin_manual_fase1["curp_fecha_desaparicion"] = np.where(
    sin_manual_fase1["curp"].isna(), np.nan,
    sin_manual_fase1["curp"].astype(str) + "_" + sin_manual_fase1["fecha_desaparicion"].astype(str),
)
EN_AMBAS = ("Registro en ambas (coincidencias administrativas sin match manual y "
            "coincidencias manuales) pero distinta FUB")
SOLO_ADMIN = "Registro solo en coincidencias administrativas sin match manual"
llaves_registros = set(registros["curp_fecha_desaparicion"].dropna())
sin_manual_fase1["coincidencia_curp_fecha"] = np.where(
    sin_manual_fase1["curp_fecha_desaparicion"].isin(llaves_registros), EN_AMBAS, SOLO_ADMIN
)
sin_manual_fase2 = sin_manual_fase1[sin_manual_fase1["coincidencia_curp_fecha"] == SOLO_ADMIN].copy()
con_manual = pd.concat([
    coincidencias_agrupado[coincidencias_agrupado["indicadora_manual"] == "con coincidencia manual"],
    sin_manual_fase1[sin_manual_fase1["coincidencia_curp_fecha"] == EN_AMBAS],
], ignore_index=True)
print(f"Solo PUI tras dos intentos: {len(sin_manual_fase2):,}")
print(f"PUI + confronta manual: {len(con_manual):,}")
snapshot("10 - conciliación PUI y manual", con_manual, mostrar=False)

## 10. Abreviaturas y unificación de PUI / confrontas manuales

Los nombres institucionales se abrevian con reglas aplicadas en secuencia. Después se juntan los casos de solo PUI y de PUI más confronta manual en una tabla común; se renombran y reducen las columnas a las necesarias para las siguientes uniones.

In [ ]:
# 10. ABREVIAR INSTITUCIONES Y FORMAR TABLA COMÚN DE COINCIDENCIAS.
for tabla in (sin_manual_fase2, con_manual):
    tabla["instituciones"] = r_str_replace_all(tabla["instituciones"], ABREVIATURAS_INSTITUCIONES)
sin_manual_fase2["indicadora_coincidencia_manual"] = "solo coincidencia en PUI"
con_manual["indicadora_coincidencia_manual"] = "coincidencia en PUI y registro manual"
combinado = pd.concat([sin_manual_fase2, con_manual], ignore_index=True)
combinado = combinado.drop(columns=[
    "indicadora_indicio_de_vida_manual", "institucion_indicadora_de_vida_manual",
    "numero_instituciones_rnpdno_manual", "indicadora_manual",
    "curp_fecha_desaparicion", "coincidencia_curp_fecha",
], errors="ignore")
combinado = combinado.rename(columns={"instituciones": "institucion_indicadora_de_vida"})
combinado["indicadora_indicio_de_vida"] = np.select(
    [combinado["indicadora_coincidencia_manual"] == "coincidencia en PUI y registro manual",
     combinado["indicadora_coincidencia_manual"] == "solo coincidencia en PUI"],
    ["con registro administrativo", "con registro administrativo en PUI"], default=None,
)
combinado = combinado.rename(columns={
    "curp": "curp_victima", "no_instituciones": "numero_instituciones", "tipo_evento": "tipo_tramites",
})[["fub", "curp_victima", "indicadora_indicio_de_vida", "institucion_indicadora_de_vida",
   "numero_instituciones", "indicadora_coincidencia_manual", "tipo_tramites", "tramites_repetidos",
   "desaparicion_activa", "fecha_evento"]]
print(f"Tabla combinada: {len(combinado):,} filas")
print(combinado["indicadora_coincidencia_manual"].value_counts().to_dict())
snapshot("11 - PUI y manual unificados", combinado)

## 11. Reintegrar registros manuales y validar FUB en RNPDNO

Los manuales que tienen indicio administrativo se cruzan de nuevo con la rama PUI por FUB y acumulan instituciones/conteos. Los casos exclusivos de PUI solo se conservan si su FUB existe en el maestro RNPDNO. Así se descartan coincidencias administrativas que no se pueden vincular a una persona de esa base.

In [ ]:
# 11. CRUZAR RAMA MANUAL Y RAMA SOLO-PUI; mantener el orden de flujo del original.
registros = registros[registros["indicadora_indicio_de_vida"] == "con registro administrativo"]
registros = registros[registros["numero_instituciones"] > 0]
lado_pui = (
    combinado[combinado["indicadora_coincidencia_manual"] == "coincidencia en PUI y registro manual"]
    .drop(columns=["curp_victima", "indicadora_indicio_de_vida"])
    .rename(columns={
        "institucion_indicadora_de_vida": "institucion_indicadora_de_vida_pui",
        "numero_instituciones": "numero_instituciones_pui",
    })
)
registros_con_pui = registros.merge(lado_pui, on="fub", how="left")
registros_con_pui = registros_con_pui.drop(columns=["curp_fecha_desaparicion"], errors="ignore")
en_ambas = registros_con_pui["indicadora_coincidencia_manual"] == "coincidencia en PUI y registro manual"
registros_con_pui["institucion_indicadora_de_vida"] = np.where(
    en_ambas,
    registros_con_pui["institucion_indicadora_de_vida"].astype(str) + ", "
        + registros_con_pui["institucion_indicadora_de_vida_pui"].astype(str),
    registros_con_pui["institucion_indicadora_de_vida"],
)
registros_con_pui["numero_instituciones"] = np.where(
    en_ambas,
    registros_con_pui["numero_instituciones"] + registros_con_pui["numero_instituciones_pui"],
    registros_con_pui["numero_instituciones"],
)
registros_con_pui = registros_con_pui.drop(columns=["institucion_indicadora_de_vida_pui", "numero_instituciones_pui"])
print(f"Rama manual tras unir instituciones PUI: {len(registros_con_pui):,} filas")
snapshot("12 - rama manual enriquecida", registros_con_pui, mostrar=False)

solo_pui = combinado[combinado["indicadora_coincidencia_manual"] == "solo coincidencia en PUI"].merge(
    comprimido[["fub", "clasificacion_des"]].assign(con_fub_en_rnpdno="con fub en rnpdno"),
    on="fub", how="left",
)
solo_pui["con_fub_en_rnpdno"] = solo_pui["con_fub_en_rnpdno"].fillna("sin fub en rnpdno")
n_con_fub = int((solo_pui["con_fub_en_rnpdno"] == "con fub en rnpdno").sum())
n_sin_fub = int((solo_pui["con_fub_en_rnpdno"] == "sin fub en rnpdno").sum())
print(f"Solo PUI con FUB en RNPDNO: {n_con_fub:,}; sin FUB: {n_sin_fub:,} (estos se descartan)")
solo_pui = solo_pui[solo_pui["con_fub_en_rnpdno"] == "con fub en rnpdno"].drop(
    columns=["con_fub_en_rnpdno", "clasificacion_des"]
)
registros_con_pui = registros_con_pui.drop(columns=["fecha_hechos_modificada"], errors="ignore")
registros_con_pui = pd.concat([registros_con_pui, solo_pui], ignore_index=True)
print(f"Ramas consolidadas: {len(registros_con_pui):,} filas")
snapshot("13 - todas las coincidencias asociadas a RNPDNO", registros_con_pui)

## 12. Filtrar registros activos y anexar defunciones

La rama de coincidencias conserva desapariciones activas y las manuales cuyo estado queda nulo, pero limita el universo RNPDNO a `Desaparecidos` y `Pendientes`. Las defunciones se anexan aparte para fechas de hechos desde el 1 de enero de 2006. Puede haber un FUB tanto en coincidencias como en defunciones: ambas filas se mantienen, como en el original.

In [ ]:
# 12. RETENER ACTIVOS/MANUALES Y AGREGAR DEFUNCIONES DEL PERÍODO ELEGIBLE.
registros_con_pui_activos = registros_con_pui[
    (registros_con_pui["desaparicion_activa"] == True) | registros_con_pui["desaparicion_activa"].isna()
].copy()
registros_con_pui_activos = registros_con_pui_activos.merge(
    comprimido[["fub", "clasificacion_des"]], on="fub", how="left"
)
registros_con_pui_activos = registros_con_pui_activos[
    registros_con_pui_activos["clasificacion_des"].isin(["Desaparecidos", "Pendientes"])
]
print(f"Coincidencias activas/manuales en Desaparecidos o Pendientes: {len(registros_con_pui_activos):,}")
snapshot("14 - activos clasificados por RNPDNO", registros_con_pui_activos, mostrar=False)

defunciones_df = defunciones.merge(
    comprimido[["fub", "curp_victima", "clasificacion_des", "fecha_hechos_modificada"]],
    on="fub", how="left",
)
defunciones_df = defunciones_df[
    defunciones_df["clasificacion_des"].isin(["Desaparecidos", "Pendientes"])
    & (defunciones_df["fecha_hechos_modificada"] >= pd.Timestamp("2006-01-01"))
]
defunciones_df["indicadora_indicio_de_vida"] = "con registro administrativo"
defunciones_df = defunciones_df.rename(columns={"tipo_defuncion_institucion": "institucion_indicadora_de_vida"})
defunciones_df["institucion_indicadora_de_vida"] = defunciones_df["institucion_indicadora_de_vida"].str.replace("_", " de ", regex=False)
defunciones_df["numero_instituciones"] = 0
defunciones_df = defunciones_df[[
    "fub", "curp_victima", "indicadora_indicio_de_vida", "institucion_indicadora_de_vida",
    "numero_instituciones", "clasificacion_des",
]]
print(f"Defunciones que cumplen universo/clasificación/fecha: {len(defunciones_df):,}")
snapshot("15 - defunciones filtradas", defunciones_df)

## 13. Salida final: diez columnas, validaciones y exportación

Se anexan coincidencias y defunciones, se eliminan campos intermedios, se aplica el contrato de diez columnas y se restituyen los tipos finales de fecha y número. La auditoría resume las dimensiones por fase; las validaciones comprueban esquema, FUB y clasificaciones antes de permitir guardar.

In [ ]:
# 13. UNION FINAL, CONTRATO DE SALIDA Y CONTROL DE CALIDAD.
registros_con_pui_activos = pd.concat([registros_con_pui_activos, defunciones_df], ignore_index=True, sort=False)
registros_con_pui_activos = registros_con_pui_activos.drop(columns=["desaparicion_activa"], errors="ignore")
registros_con_pui_activos["tipo_tramites"] = r_str_replace_all(
    registros_con_pui_activos["tipo_tramites"], ABREVIATURAS_INSTITUCIONES
)

for columna in COLUMNAS_SALIDA:
    if columna not in registros_con_pui_activos:
        registros_con_pui_activos[columna] = pd.NA
registros_con_pui_activos = registros_con_pui_activos[COLUMNAS_SALIDA].copy()
registros_con_pui_activos["fecha_evento"] = pd.to_datetime(registros_con_pui_activos["fecha_evento"], errors="coerce")
registros_con_pui_activos["numero_instituciones"] = pd.to_numeric(
    registros_con_pui_activos["numero_instituciones"], errors="coerce"
).astype("Int64")

audit_final = snapshot("16 - salida final PUI", registros_con_pui_activos)
problemas = []
if list(registros_con_pui_activos.columns) != COLUMNAS_SALIDA:
    problemas.append("El orden/esquema final no coincide con las diez columnas declaradas.")
n_fub_nulos = int(registros_con_pui_activos["fub"].isna().sum())
if n_fub_nulos:
    problemas.append(f"Hay {n_fub_nulos:,} FUB nulos.")
es_defuncion = registros_con_pui_activos["institucion_indicadora_de_vida"].astype("string").str.startswith("defuncion de", na=False)
dup_no_def = int(registros_con_pui_activos.loc[~es_defuncion, "fub"].duplicated().sum())
dup_def = int(registros_con_pui_activos.loc[es_defuncion, "fub"].duplicated().sum())
if dup_no_def + dup_def:
    problemas.append(f"Hay {dup_no_def + dup_def:,} duplicados dentro de una misma rama.")
solapes = len(set(registros_con_pui_activos.loc[es_defuncion, "fub"].dropna())
              & set(registros_con_pui_activos.loc[~es_defuncion, "fub"].dropna()))
if solapes:
    print(f"Aviso informativo: {solapes:,} FUB figuran en una rama PUI/manual y en defunciones; ambas filas se conservan.")
clases = set(registros_con_pui_activos["clasificacion_des"].dropna().unique())
if not clases <= {"Desaparecidos", "Pendientes"}:
    problemas.append(f"Clasificaciones inesperadas: {clases - {'Desaparecidos', 'Pendientes'}}")
if problemas:
    for problema in problemas:
        warnings.warn(problema)
else:
    print("Validaciones de salida superadas.")

print(f"\nCorte procesado: {FECHA_CORTE}")
print(f"Filas finales: {len(registros_con_pui_activos):,} × {len(COLUMNAS_SALIDA)} columnas")
print("Por clasificación:")
display(registros_con_pui_activos["clasificacion_des"].value_counts(dropna=False).to_frame("filas"))
print("Por indicadora de coincidencia manual:")
display(registros_con_pui_activos["indicadora_coincidencia_manual"].value_counts(dropna=False).to_frame("filas"))
print("\nResumen de evolución de DataFrames:")
display(pd.DataFrame(AUDITORIA))

if EXPORTAR_RESULTADO:
    RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)
    salida_exportar = registros_con_pui_activos.rename(columns=NOMBRES_EXPORTADOS)
    salida_exportar.to_parquet(RUTA_SALIDA, index=False, engine="pyarrow")
    print(f"Archivo Parquet exportado: {RUTA_SALIDA}")
else:
    print("Exportación apagada. Para escribir el Parquet, activa EXPORTAR_RESULTADO en la primera sección.")

## 14. Comparación opcional con la salida PUI ya existente

Si existe el Parquet normal del proyecto, se compara con la tabla actual tras aplicar los mismos nombres exportados. La comparación reporta filas, esquema y diferencias por columna, sin imprimir FUB ni CURP. Sirve para verificar si la réplica autónoma es equivalente antes de exportarla.

In [ ]:
# 14. COMPARAR CON EL PARQUET PERSISTIDO DEL PIPELINE NORMAL, SIN MODIFICARLO.
RUTA_SALIDA_REFERENCIA = PROJECT_ROOT / "output" / "Resultado_3_Procesamiento_ETL_PUI.parquet"
if not RUTA_SALIDA_REFERENCIA.exists():
    print(f"No hay Parquet previo para comparar: {RUTA_SALIDA_REFERENCIA}")
else:
    actual_exportado = registros_con_pui_activos.rename(columns=NOMBRES_EXPORTADOS)
    previo = pd.read_parquet(RUTA_SALIDA_REFERENCIA)
    columnas_comunes = [c for c in actual_exportado.columns if c in previo.columns]
    print(f"Referencia: {RUTA_SALIDA_REFERENCIA.name} {previo.shape}; autónomo {actual_exportado.shape}")
    print("Columnas que difieren:", {
        "solo_referencia": sorted(set(previo.columns) - set(actual_exportado.columns)),
        "solo_autonomo": sorted(set(actual_exportado.columns) - set(previo.columns)),
    })
    diferencias = {}
    if len(previo) == len(actual_exportado):
        for columna in columnas_comunes:
            izq = previo[columna].astype("string").fillna("<NA>").reset_index(drop=True)
            der = actual_exportado[columna].astype("string").fillna("<NA>").reset_index(drop=True)
            n_distintos = int(izq.ne(der).sum())
            if n_distintos:
                diferencias[columna] = n_distintos
        if diferencias:
            print("Celdas con diferencias por columna (orden de fila):")
            display(pd.Series(diferencias, name="celdas_distintas").sort_values(ascending=False).to_frame())
        else:
            print("Valores coinciden en las columnas compartidas, fila por fila.")
    else:
        print("No se pueden alinear valores por fila: las cantidades son diferentes.")
    mismo_esquema = list(previo.columns) == list(actual_exportado.columns)
    mismos_registros = len(previo) == len(actual_exportado)
    salida_identica = mismo_esquema and mismos_registros and not diferencias
    print(f"¿Mismas columnas/orden, filas y valores? {salida_identica}")

In [ ]:
# Exportación adicional solicitada: archivo CSV de prueba en la carpeta output/.
RUTA_PRUEBA_CSV = PROJECT_ROOT / "output" / "Prueba 3Luis.csv"
RUTA_PRUEBA_CSV.parent.mkdir(parents=True, exist_ok=True)
salida_exportar = registros_con_pui_activos.rename(columns=NOMBRES_EXPORTADOS)
salida_exportar.to_csv(RUTA_PRUEBA_CSV, index=False, encoding="utf-8-sig")
print(f"CSV de prueba exportado: {RUTA_PRUEBA_CSV} ({len(salida_exportar):,} filas)")